# ForgeAI — TWF 일관성 5회 측정 (이슈 #43)

**목적**: `prompts/action_plan_v1.py` TWF addendum 수정 후 일관성 재측정  
**환경**: Colab T4 GPU  
**측정**: `--runs 5 --samples 6` (6 층화 × 5회 = 30회 파이프라인)

> 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행

## 0. GPU 확인

In [ ]:
!nvidia-smi

## 1. Ollama 설치 및 시작

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time, requests

# Ollama 서버 백그라운드 시작
proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# 서버 준비 대기 (최대 30초)
for i in range(30):
    try:
        r = requests.get("http://localhost:11434", timeout=2)
        if r.status_code == 200:
            print(f"Ollama 서버 준비 완료 ({i+1}초)")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama 서버 시작 실패")

## 2. 모델 풀 (qwen2.5:7b + nomic-embed-text)

In [ ]:
# qwen2.5:7b — 약 4.7GB, T4 VRAM(16GB)에 여유 있게 로드됨
!ollama pull qwen2.5:7b

In [ ]:
# 임베딩 모델
!ollama pull nomic-embed-text

## 3. ForgeAI 저장소 클론 & 의존성 설치

In [ ]:
import os

REPO = "https://github.com/enjoylonelines/ForgeAI.git"
BRANCH = "fix/43-twf-consistency-fail"   # 수정이 반영된 브랜치
WORKDIR = "/content/ForgeAI"

!git clone --depth=1 --branch {BRANCH} {REPO} {WORKDIR}
os.chdir(WORKDIR)
print("작업 디렉토리:", os.getcwd())

In [ ]:
# pip로 의존성 설치 (uv 없이)
!pip install -q \
    chromadb \
    langchain \
    langchain-ollama \
    langchain-chroma \
    langgraph \
    pydantic \
    python-dotenv \
    pypdf \
    pandas \
    ucimlrepo \
    scikit-learn \
    xgboost \
    httpx \
    requests \
    structlog
print("설치 완료")

## 4. AI4I 데이터셋 다운로드

In [ ]:
import os
from pathlib import Path

os.makedirs("data", exist_ok=True)
data_csv = Path("data/ai4i2020.csv")

if not data_csv.exists():
    print("UCI 저장소에서 AI4I 2020 데이터셋 다운로드 중...")
    from ucimlrepo import fetch_ucirepo
    import pandas as pd

    dataset = fetch_ucirepo(id=601)
    X = dataset.data.features.copy()
    y = dataset.data.targets.copy()
    df = pd.concat([X, y], axis=1)
    df.to_csv(data_csv, index=False)
    print(f"저장 완료: {data_csv} ({len(df)}행)")
else:
    print(f"이미 존재: {data_csv}")

## 5. SOP 문서 ChromaDB 인덱싱

In [ ]:
import asyncio
import sys
sys.path.insert(0, "/content/ForgeAI")

from pathlib import Path
from rag.ingestion import ingest_document
from rag.chroma_client import get_sop_collection

SOP_DIR = Path("/content/ForgeAI/data/sop_docs")

async def index_sops():
    collection = get_sop_collection()
    if collection.count() > 0:
        print(f"이미 인덱싱됨: {collection.count()}개 청크")
        return

    sop_files = sorted(SOP_DIR.glob("*.md"))
    print(f"SOP 파일 {len(sop_files)}개 인덱싱 시작...")

    for sop_file in sop_files:
        result = await ingest_document(
            file_bytes=sop_file.read_bytes(),
            filename=sop_file.name,
            content_type="text/markdown",
        )
        print(f"  ✓ {sop_file.name}: {result.chunk_count}청크 추가 (총 {result.collection_total})")

    print(f"\n인덱싱 완료 — 총 {get_sop_collection().count()}개 청크")

await index_sops()

## 6. 일관성 프로토콜 실행 (5회 × 6층화 = 30회)

In [ ]:
# 출력 파일 경로
REPORT_OUT = "/content/consistency_report_post_fix.md"

!cd /content/ForgeAI && python scripts/consistency_protocol.py \
    --runs 5 \
    --samples 6 \
    --out {REPORT_OUT} \
    2>&1 | grep -v '^{"timestamp"'  # JSON 로그 필터링

## 7. 결과 확인

In [ ]:
from pathlib import Path

report = Path(REPORT_OUT)
if report.exists():
    print(report.read_text())
else:
    print("리포트 파일이 생성되지 않았습니다. 위 셀 출력을 확인하세요.")

## 8. 결과 파일 다운로드

리포트를 로컬로 다운로드하여 `docs/consistency_report.md`를 업데이트하세요.

In [ ]:
from google.colab import files

if Path(REPORT_OUT).exists():
    files.download(REPORT_OUT)
else:
    print("다운로드할 파일이 없습니다.")